In [51]:
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "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"
os.environ["LANGSMITH_PROJECT"] = "rag"
os.environ["GROQ_API_KEY"] = "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"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"

In [ ]:
import os
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:llama-3.3-70b-versatile")

In [34]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [35]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db_agent",  # Where to save data locally, remove if not necessary
)

In [36]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Only keep post title, headers, and content from the full HTML.
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

Total characters: 43047


In [37]:
print(docs[0].page_content[:500])



      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In


In [38]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 63 sub-documents.


In [39]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['60b3a33c-f8a2-4482-9e69-3ca20e725c39', 'bb87e2dd-6029-4439-9eb4-a72f311ea395', '6bc7ca67-4d61-40db-ba1a-7a9c9c285c25']


In [40]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [59]:
def retrieve_context(query):
    docs = vector_store.similarity_search(query, k=4)

    if not docs:
        return ""

    return "\n\n".join(doc.page_content for doc in docs)


In [66]:
def build_prompt(context, question):
    return f"""
You are a QA system that ONLY uses the provided context.

If the answer is not in the context, say:
"I don't know based on the document."

Context:
{context}

Question:
{question}

Answer:
"""


In [67]:
def ask_rag(question):
    context = retrieve_context(question)
    prompt = build_prompt(context, question)

    response = model.invoke(prompt)
    return response.content


In [68]:
query = "What is the standard method for Task Decomposition?"
print(ask_rag(query))

Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks.


In [69]:
print(ask_rag("What is quantum theory?"))

I don't know based on the document.


In [77]:
print(ask_rag("What is the Case Study Scientific Discovery Agent about in this document?"))

The Case Study Scientific Discovery Agent, as mentioned in the context by Boiko et al. (2023), is about LLM-empowered agents for scientific discovery. It is designed to handle autonomous design, planning, and performance of complex scientific experiments. For example, when given the task to "develop a novel anticancer drug", the agent performed the following steps: inquired about current trends, selected a target, requested a scaffold, and attempted synthesis of the identified compound. The agent can use various tools such as browsing the internet, reading documentation, executing code, and calling robotics experimentation APIs, as well as leveraging other LLMs.
